# CGFL: GPU re-run check (reproducibility)

Re-runs two archived configurations from scratch (CGFL and DP-FedAvg, record-level DP, eps = 3, seed 1) and compares them with the archived run files.

**Inputs (Add Input):** `cgfl-code` (latest version), `ptbxl-prep` (ptbxl100.npz), and the latest output of the experiments notebook (`cgfl_kaggle_rev4`, which holds all run files).

**Accelerator:** GPU T4 x2. Run interactively (Run All) or commit; about 15 minutes.

In [ ]:
import os, glob, shutil, subprocess, json
FOUND = {"cgfl.py": [], "ptbxl100.npz": [], "json": []}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    dirs[:] = [d for d in dirs if d not in {"records100", "records500", "records"}]
    for f in files:
        p = os.path.join(root, f)
        if f in FOUND: FOUND[f].append(p)
        elif os.path.basename(root) == "results" and f.endswith(".json"): FOUND["json"].append(p)
print({k: len(v) for k, v in FOUND.items()})
code = [p for p in FOUND["cgfl.py"] if "labelset" in open(p).read() and "ceil(math.log2(d))" in open(p).read()]
assert code, "upload the newest cgfl.py (18-bit index cost, lr_dp default 0.5) as a New Version of cgfl-code"
os.chdir("/kaggle/working"); shutil.copy(code[0], "cgfl.py")
os.makedirs("archived", exist_ok=True)
for p in FOUND["json"]: shutil.copy(p, "archived")
DATA = FOUND["ptbxl100.npz"][0]
print("archived runs:", len(os.listdir("archived")), "| data:", DATA)

In [ ]:
# two re-runs in parallel, one per GPU (about 12 minutes)
cmds = [f"python cgfl.py run --data {DATA} --method capacity --dp record --mask public --seed 1 --results rerun --ckpt rerun_ck --device cuda:0",
        f"python cgfl.py run --data {DATA} --method fedavg --dp record --seed 1 --results rerun --ckpt rerun_ck --device cuda:1"]
procs = [subprocess.Popen(c, shell=True, stdout=open(f"rerun_{i}.log", "w"), stderr=subprocess.STDOUT) for i, c in enumerate(cmds)]
for p in procs: p.wait()
for i in range(2): print(open(f"rerun_{i}.log").read().strip().splitlines()[-1])

In [ ]:
%%writefile repro_check.py
#!/usr/bin/env python3
"""Compare freshly re-run results with the archived ones (same run IDs).

Usage: python repro_check.py <archived results dir> <re-run results dir>
GPU kernels are not bit-wise deterministic, so AUC is compared with a tolerance; privacy (sigma),
data volume and uplink time are deterministic given the seed and must match.
Archived Top-k runs stored 32-bit indices; they are rescaled to ceil(log2 d)-bit indices as in the paper.
"""
import sys, os, json, glob, math

OLD, NEW = sys.argv[1:3]
TOL_AUC = 0.02
bad = 0
for f in sorted(glob.glob(os.path.join(NEW, "*.json"))):
    rid = os.path.basename(f)
    g = os.path.join(OLD, rid)
    if not os.path.exists(g):
        print(f"{rid}: no archived counterpart"); continue
    n, o = json.load(open(f)), json.load(open(g))
    c = o["cfg"]
    fac = 1.0
    if c.get("mask") == "topk" and c.get("method") not in ("fedavg", "qsgd", "qsgd_cap"):
        fac = (32 + math.ceil(math.log2(o["d"]))) / 64.0
    checks = [("test AUC", n["test"]["auc"], o["test"]["auc"], TOL_AUC),
              ("sigma", n["sigma_mean"], o["sigma_mean"], 1e-6),
              ("data MB", n["comm_mb"], fac * o["comm_mb"], 1e-3 * max(1, o["comm_mb"])),
              ("uplink s", n["wallclock_uplink_s"], fac * o["wallclock_uplink_s"], 1e-3 * max(1, o["wallclock_uplink_s"]))]
    for name, a, b, tol in checks:
        ok = abs(a - b) <= tol
        bad += not ok
        print(f"{rid[:-5]} {c.get('method'):9s} dp={c.get('dp'):6s} seed={c.get('seed')}  {name:9s} "
              f"re-run={a:10.4f}  archived={b:10.4f}  {'OK' if ok else 'DIFFERS'}")
print("\nall checks passed" if bad == 0 else f"\n{bad} check(s) differ")


In [ ]:
!python repro_check.py archived rerun
shutil.make_archive("rerun_results", "zip", "rerun")